In [0]:
# Mount ADLS Gen2
# Required each time the cluster is restarted which should be only on the first notebook as they run in order

tiers = ["bronze", "silver", "gold"]
adls_paths = {tier: f"abfss://{tier}@adlsnikhildev.dfs.core.windows.net/" for tier in tiers}

# Accessing paths
bronze_adls = adls_paths["bronze"]
silver_adls = adls_paths["silver"]
gold_adls = adls_paths["gold"] 

print(dbutils.fs.ls(bronze_adls))
print(dbutils.fs.ls(silver_adls))
print(dbutils.fs.ls(gold_adls))

[FileInfo(path='abfss://bronze@adlsnikhildev.dfs.core.windows.net/2026-06-05_earthquake_data.json', name='2026-06-05_earthquake_data.json', size=427152, modificationTime=1780731273000)]
[]
[]


[FileInfo(path='abfss://bronze@adlsnikhildev.dfs.core.windows.net/2026-06-05_earthquake_data.json', name='2026-06-05_earthquake_data.json', size=427152, modificationTime=1780731273000)]

In [0]:
# from datetime import date, timedelta
# start_date = date.today() - timedelta(days=1)
# end_date = date.today()

In [0]:

try:
    bronze_output = dbutils.jobs.taskValues.get(
        taskKey="bronze",
        key="bronze_output"
    )
except:
    bronze_output = {
        "start_date": "2026-06-05",
        "end_date": "2026-06-06",
        "bronze_path": adls_paths["bronze"],
        "silver_path": adls_paths["silver"],
        "gold_path": adls_paths["gold"]
    }

start_date = bronze_output.get("start_date", "")
end_date = bronze_output.get("end_date", "")

bronze_adls = bronze_output.get("bronze_path", "")
silver_adls = bronze_output.get("silver_path", "")

print(f"Start date: {start_date}")
print(f"End date: {end_date}")
print(f"Bronze ADLS: {bronze_adls}")
print(f"Silver ADLS: {silver_adls}")

Start date: 2026-06-05
End date: 2026-06-06
Bronze ADLS: abfss://bronze@adlsnikhildev.dfs.core.windows.net/
Silver ADLS: abfss://silver@adlsnikhildev.dfs.core.windows.net/


In [0]:
from pyspark.sql.functions import col, isnull, when
from pyspark.sql.types import TimestampType
from datetime import date, timedelta

# Load the JSON data into a Spark DataFrame
df = spark.read.option("multiline", "true").json(f"{bronze_adls}{start_date}_earthquake_data.json")


In [0]:
display(df.head())

Row(geometry=Row(coordinates=[-155.7431640625, 19.5060005187988, 6.03000020980835], type='Point'), id='hv74977022', properties=Row(alert=None, cdi=None, code='74977022', detail='https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=hv74977022&format=geojson', dmin=0.03817, felt=None, gap=121, ids=',hv74977022,', mag=1.94, magType='md', mmi=None, net='hv', nst=30, place='13 km ENE of Honaunau-Napoopoo, Hawaii', rms=0.280000001, sig=58, sources=',hv,', status='automatic', time=1780703997200, title='M 1.9 - 13 km ENE of Honaunau-Napoopoo, Hawaii', tsunami=0, type='earthquake', types=',origin,phase-data,', tz=None, updated=1780704259380, url='https://earthquake.usgs.gov/earthquakes/eventpage/hv74977022'), type='Feature')

In [0]:
# display(df)

In [0]:

# Reshape earthquake data
df = (
    df
    .select(
        'id',
        col('geometry.coordinates').getItem(0).alias('longitude'),
        col('geometry.coordinates').getItem(1).alias('latitude'),
        col('geometry.coordinates').getItem(2).alias('elevation'),
        col('properties.title').alias('title'),
        col('properties.place').alias('place_description'),
        col('properties.sig').alias('sig'),
        col('properties.mag').alias('mag'),
        col('properties.magType').alias('magType'),
        col('properties.time').alias('time'),
        col('properties.updated').alias('updated')
    )
)

In [0]:
display(df)

id,longitude,latitude,elevation,title,place_description,sig,mag,magType,time,updated
hv74977022,-155.7431640625,19.5060005187988,6.03000020980835,"M 1.9 - 13 km ENE of Honaunau-Napoopoo, Hawaii","13 km ENE of Honaunau-Napoopoo, Hawaii",58,1.94,md,1780703997200,1780704259380
tx2026lajfxs,-102.03,31.782,2.2703,"M 1.3 - 24 km S of Midland, Texas","24 km S of Midland, Texas",26,1.3,ml,1780703903389,1780704163495
nc75371591,-122.847503662109,38.8466682434082,1.46000003814697,"M 0.7 - 11 km NW of The Geysers, CA","11 km NW of The Geysers, CA",7,0.69,md,1780703672510,1780703770790
nc75371586,-122.807830810547,38.8368339538574,1.89999997615814,"M 0.7 - 8 km WNW of Cobb, CA","8 km WNW of Cobb, CA",8,0.74,md,1780703666740,1780703760862
nc75371581,-122.844169616699,38.8501663208008,2.9300000667572,"M 0.3 - 11 km WNW of Cobb, CA","11 km WNW of Cobb, CA",1,0.25,md,1780703171290,1780703265621
av94227028,-169.878833333333,52.8426666666667,9.69,"M 0.9 - 69 km W of Nikolski, Alaska","69 km W of Nikolski, Alaska",13,0.93,ml,1780702898690,1780723940350
nc75371571,-122.847503662109,38.8463325500488,1.53999996185303,"M 0.4 - 11 km NW of The Geysers, CA","11 km NW of The Geysers, CA",2,0.38,md,1780702754950,1780702850766
tx2026laipil,-101.623,32.073,2.9226,"M 1.2 - 16 km ESE of Stanton, Texas","16 km ESE of Stanton, Texas",22,1.2,ml,1780702748252,1780704084556
av94227023,-169.8635,52.8228333333333,8.52,"M 1.2 - 68 km W of Nikolski, Alaska","68 km W of Nikolski, Alaska",23,1.23,ml,1780702539130,1780723759980
nc75371561,-122.84716796875,38.8468322753906,1.5,"M 0.4 - 11 km NW of The Geysers, CA","11 km NW of The Geysers, CA",3,0.42,md,1780701782110,1780701880859


In [0]:

# Validate data: Check for missing or null values
df = (
    df
    .withColumn('longitude', when(isnull(col('longitude')), 0).otherwise(col('longitude')))
    .withColumn('latitude', when(isnull(col('latitude')), 0).otherwise(col('latitude')))
    .withColumn('time', when(isnull(col('time')), 0).otherwise(col('time')))
)

In [0]:

# Convert 'time' and 'updated' to timestamp from Unix time
df = (
    df
    .withColumn('time', (col('time') / 1000).cast(TimestampType()))
    .withColumn('updated', (col('updated') / 1000).cast(TimestampType()))
)

In [0]:
display(df)

id,longitude,latitude,elevation,title,place_description,sig,mag,magType,time,updated
hv74977022,-155.7431640625,19.5060005187988,6.03000020980835,"M 1.9 - 13 km ENE of Honaunau-Napoopoo, Hawaii","13 km ENE of Honaunau-Napoopoo, Hawaii",58,1.94,md,2026-06-05T23:59:57.200Z,2026-06-06T00:04:19.380Z
tx2026lajfxs,-102.03,31.782,2.2703,"M 1.3 - 24 km S of Midland, Texas","24 km S of Midland, Texas",26,1.3,ml,2026-06-05T23:58:23.389Z,2026-06-06T00:02:43.495Z
nc75371591,-122.847503662109,38.8466682434082,1.46000003814697,"M 0.7 - 11 km NW of The Geysers, CA","11 km NW of The Geysers, CA",7,0.69,md,2026-06-05T23:54:32.510Z,2026-06-05T23:56:10.790Z
nc75371586,-122.807830810547,38.8368339538574,1.89999997615814,"M 0.7 - 8 km WNW of Cobb, CA","8 km WNW of Cobb, CA",8,0.74,md,2026-06-05T23:54:26.740Z,2026-06-05T23:56:00.862Z
nc75371581,-122.844169616699,38.8501663208008,2.9300000667572,"M 0.3 - 11 km WNW of Cobb, CA","11 km WNW of Cobb, CA",1,0.25,md,2026-06-05T23:46:11.290Z,2026-06-05T23:47:45.621Z
av94227028,-169.878833333333,52.8426666666667,9.69,"M 0.9 - 69 km W of Nikolski, Alaska","69 km W of Nikolski, Alaska",13,0.93,ml,2026-06-05T23:41:38.690Z,2026-06-06T05:32:20.350Z
nc75371571,-122.847503662109,38.8463325500488,1.53999996185303,"M 0.4 - 11 km NW of The Geysers, CA","11 km NW of The Geysers, CA",2,0.38,md,2026-06-05T23:39:14.950Z,2026-06-05T23:40:50.766Z
tx2026laipil,-101.623,32.073,2.9226,"M 1.2 - 16 km ESE of Stanton, Texas","16 km ESE of Stanton, Texas",22,1.2,ml,2026-06-05T23:39:08.252Z,2026-06-06T00:01:24.556Z
av94227023,-169.8635,52.8228333333333,8.52,"M 1.2 - 68 km W of Nikolski, Alaska","68 km W of Nikolski, Alaska",23,1.23,ml,2026-06-05T23:35:39.130Z,2026-06-06T05:29:19.980Z
nc75371561,-122.84716796875,38.8468322753906,1.5,"M 0.4 - 11 km NW of The Geysers, CA","11 km NW of The Geysers, CA",3,0.42,md,2026-06-05T23:23:02.110Z,2026-06-05T23:24:40.859Z


In [0]:

# Save the transformed DataFrame to the Silver container
silver_output_path = f"{silver_adls}earthquake_events_silver/"

In [0]:

# Append DataFrame to Silver container in Parquet format
df.write.mode('append').parquet(silver_output_path)

In [0]:
# Save the transformed DataFrame to the Silver container
silver_output_path = f"{silver_adls}earthquake_events_silver/"


# Append DataFrame to Silver container in Parquet format
df.write.mode('append').parquet(silver_output_path)

# Set task values for workflow
dbutils.jobs.taskValues.set(key="silver_output", value=silver_output_path)
print("Silver notebook completed successfully!")



Silver notebook completed successfully!
